# PCA Factor Model + VAR Prediction — All Rebalance Years (2006–2025)

---

## Overview

Notebooks 02 and 03 predict Russell membership by fitting **univariate** time-series models on each stock's own market cap history. That approach has a structural ceiling: each stock is modelled in isolation, so the predictions are **cross-sectionally incoherent** — there is no mechanism to ensure that the predicted cap of stock A is consistent with the predicted cap of stock B, even if A and B move together every day.

This notebook replaces the per-stock univariate approach with a **cross-sectional factor model**:

1. **PCA on the return cross-section**: each trading day, all stock log-returns form one row of a T × N matrix. PCA finds the K directions of maximum co-movement. The first few PCs are interpretable as market, size, and sector factors.
2. **Bai-Ng IC1 for K selection**: the number of factors K is chosen by the Bai & Ng (2002) panel information criterion — the only principled BIC-type criterion designed for large-panel factor models. This replaces the ad-hoc '90% variance explained' rule.
3. **VAR(p) on factor scores**: the K factor score time series are modelled jointly by a Vector Autoregression. The lag order p is selected by VAR BIC. This captures cross-factor dynamics (e.g. the market factor leading the size factor).
4. **Reconstruction → ranking**: the VAR's 60-step forecast of factor scores is projected back through the PCA loadings to recover predicted cumulative log-returns for every stock. Combined with the last observed market cap, this gives a predicted market cap for each stock → rank → evaluate Russell membership.

---

## Methodology

| Parameter | Value | Rationale |
|---|---|---|
| PCA input | Daily log-returns from price data | Returns are stationary; PCA on levels is not meaningful |
| K selection | **Bai-Ng IC1** | Panel IC designed for large T, large N factor models |
| Factor score model | **VAR(p)**, p selected by VAR BIC | Joint dynamics; standard multivariate time-series baseline |
| t = 0 | Russell Rank Date (~late April / May) | Same as notebooks 02/03 |
| Training window | Up to 315 trading days before rank date | ~15 months; same as notebooks 02/03 |
| Forecast horizon | **60 trading days** after rank date | ~3 calendar months; same as notebooks 02/03 |
| Coverage filter | ≥ 80% non-NaN in training window | Excludes thin-history tickers from PCA |
| PCA type | Covariance-based (demean only, no scaling) | Preserves return variance heterogeneity across stocks |

### Why Bai-Ng IC1?

Standard BIC compares models on the same data. When K changes, the scores matrix changes, making direct BIC comparison across K invalid. Bai & Ng (2002) derive a penalty specifically calibrated for T × N panels:

$$IC_1(k) = \ln\hat{\sigma}^2_k + k \cdot \frac{N+T}{NT} \ln\frac{NT}{N+T}$$

where $\hat{\sigma}^2_k$ is the mean squared residual of the k-factor reconstruction. The penalty grows with k and shrinks as N, T grow — avoiding over-selection in large panels.

### Expected Results

- **K\*** will likely be 5–20 factors (the market factor alone explains ~30–40%; getting to diminishing returns takes ~10–20).
- **Spearman ρ** should be similar to notebooks 02/03 (~0.97–0.99) — the size hierarchy is persistent regardless of model.
- **MAE Rank** is the key test: if the PCA+VAR is better at capturing cross-sectional co-movement, MAE should be lower in volatile years (2009, 2020, 2022) where factor shocks dominated individual stock moves.
- **Add/remove recall** remains structurally low — boundary-crossing stocks are outliers by definition, and no factor model can predict idiosyncratic outlier moves.

---

## 1. Setup

In [ ]:
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from statsmodels.tsa.vector_ar.var_model import VAR

sns.set_theme(style='whitegrid', font_scale=1.05)
warnings.filterwarnings('ignore')

# ── Data paths ────────────────────────────────────────────────────────────────
PROCESSED_DIR  = Path('../../data/processed')
RAW_MBR_DIR    = Path('../../data/raw/membership')

PRICES_PARQUET = PROCESSED_DIR / 'FINALIZED_PRICES.parquet'
MCAPS_PARQUET  = PROCESSED_DIR / 'FINALIZED_MCAPS.parquet'
CALENDAR_FILE  = RAW_MBR_DIR   / 'russell_calendar.xlsx'
CACHE_DIR      = PROCESSED_DIR / 'pca_var_cache'
CACHE_DIR.mkdir(exist_ok=True)

# ── Model parameters ─────────────────────────────────────────────────────────
TRAIN_DAYS   = 315    # max training window (trading days)
STEPS        = 60     # forecast horizon (trading days, ~3 calendar months)
MAX_K        = 30     # max factors tested in Bai-Ng IC
MAX_VAR_LAGS = 5      # max VAR lag order tested by BIC
MIN_COVERAGE = 0.80   # min non-NaN fraction in training window to include ticker in PCA
MIN_OBS_MCAP = 40     # min mcap observations required for a ticker to enter final ranking

print(f'Forecast horizon : {STEPS} trading days')
print(f'Training window  : up to {TRAIN_DAYS} trading days')
print(f'Max K (Bai-Ng)   : {MAX_K}')
print(f'Max VAR lags     : {MAX_VAR_LAGS}')
print(f'Coverage filter  : ≥ {MIN_COVERAGE:.0%} non-NaN')

---

## 2. Data Loading

### 2.1 Prices → Log-Returns

PCA requires a **stationary** cross-section. Daily log-returns satisfy this: $r_{i,t} = \ln P_{i,t} - \ln P_{i,t-1}$. The full T × N return matrix is pre-computed once here; the main loop slices training windows from it on the fly.

In [ ]:
prices = pd.read_parquet(PRICES_PARQUET)
log_returns = np.log(prices).diff()   # T × N; first row is NaN by construction

print(f'Prices matrix    : {prices.shape[0]:,} dates × {prices.shape[1]:,} tickers')
print(f'Date range       : {prices.index.min().date()} → {prices.index.max().date()}')
print(f'Return coverage  : {log_returns.notna().mean().mean():.1%} of cells non-NaN')

### 2.2 Market Cap Matrix

Market caps are used for two purposes only:
- **Reconstruction**: multiply the last observed mcap by `exp(cumulative predicted log-return)` to get predicted market cap
- **Evaluation**: actual mcap at the 60-day target date defines the ground-truth rank

In [ ]:
mcaps = pd.read_parquet(MCAPS_PARQUET)
print(f'Market cap matrix: {mcaps.shape[0]:,} dates × {mcaps.shape[1]:,} tickers')
print(f'Date range       : {mcaps.index.min().date()} → {mcaps.index.max().date()}')
print(f'Coverage         : {mcaps.notna().mean().mean():.1%} of cells non-NaN')

### 2.3 Russell Rebalance Calendar

Identical to notebooks 02/03. The **Rank Date** is `t = 0`; the forecast target is the 60th trading day after it (~early August each year).

In [ ]:
_cal_raw = pd.read_excel(CALENDAR_FILE, sheet_name='Russell Calendar', header=3)
_cal_raw.columns = ['Year', 'Period', 'Rank_Date', 'Announcement_Date',
                    'Effective_Date', 'Effective_Open', 'Notes']
calendar = (
    _cal_raw[['Year', 'Rank_Date', 'Effective_Date']]
    .dropna(subset=['Year'])
    .assign(
        Year           = lambda d: d['Year'].astype(int),
        Rank_Date      = lambda d: pd.to_datetime(d['Rank_Date']),
        Effective_Date = lambda d: pd.to_datetime(d['Effective_Date']),
    )
    .set_index('Year')
)
print(f'Calendar: {len(calendar)} years  ({calendar.index.min()} – {calendar.index.max()})')
calendar

---

## 3. Model Definition

### 3.1 Bai-Ng IC1 — Factor Count Selection

Standard BIC is not valid for comparing factor models with different K because changing K changes the data (the scores matrix). Bai & Ng (2002) derive a penalty calibrated for T × N panels. IC1 is the most widely used variant:

$$IC_1(k) = \ln\hat{\sigma}^2_k + k \cdot \frac{N+T}{NT} \ln\frac{NT}{N+T}$$

As k increases, the fit improves (σ² falls) but the penalty grows. The minimum IC1 gives K*.

In [ ]:
def bai_ng_ic1(X, max_k):
    """
    Bai & Ng (2002) IC1 criterion for number of PCA factors.
    X : T × N demeaned return array (no NaNs)
    Returns K* in [1, max_k] that minimises IC1.
    """
    T, N = X.shape
    actual_max_k = min(max_k, T - 1, N - 1)
    pca = PCA(n_components=actual_max_k)
    pca.fit(X)

    # Penalty constant shared across k values
    penalty_const = (N + T) / (N * T) * np.log(N * T / (N + T))

    ics = []
    for k in range(1, actual_max_k + 1):
        # Reconstruct with k factors and compute mean squared residual
        scores_k = pca.transform(X)[:, :k]       # T × k
        recon_k  = scores_k @ pca.components_[:k] # T × N
        sigma_k  = np.mean((X - recon_k) ** 2)
        ics.append(np.log(sigma_k) + k * penalty_const)

    return int(np.argmin(ics)) + 1   # 1-indexed

### 3.2 VAR(p) Forecaster

PC scores from PCA on returns are already stationary (they inherit stationarity from returns), so no differencing is needed. We fit a VAR(p) directly on the K* score time series:

$$\mathbf{f}_t = \mathbf{c} + \sum_{j=1}^{p} \mathbf{A}_j \mathbf{f}_{t-j} + \boldsymbol{\varepsilon}_t$$

where $\mathbf{f}_t \in \mathbb{R}^{K^*}$ is the vector of factor scores on day t.

The lag order p* is selected by VAR BIC via `statsmodels.VAR.select_order()`. We enforce `p* ≥ 1`. Forecasts are generated iteratively using `fitted.forecast()`.

In [ ]:
def fit_var_and_forecast(scores, max_lags, steps):
    """
    Fit VAR(p*) on scores (T × K), p* selected by BIC.
    Returns (forecast array of shape steps × K, p*).
    """
    model   = VAR(scores)
    lag_res = model.select_order(maxlags=max_lags)
    p_star  = lag_res.selected_orders.get('bic', 1)
    p_star  = max(int(p_star), 1)          # at least VAR(1)
    fitted  = model.fit(p_star)
    # forecast needs the last p_star rows as initial values
    fc = fitted.forecast(scores[-p_star:], steps=steps)  # steps × K
    return fc, p_star

### 3.3 Single-Year Pipeline

For each rebalance year:

1. Slice training log-returns (up to 315 days ending at rank date)
2. Coverage filter: keep tickers with ≥ 80% non-NaN; fill remaining gaps with 0
3. Demean columns (covariance-based PCA)
4. Bai-Ng IC1 → K*
5. Fit PCA(K*) → extract scores (T × K*) and loadings (K* × N)
6. VAR(p*) on scores → 60-step forecast
7. Reconstruct: `cumulative_logret = (forecast @ loadings).sum(axis=0)` → shape (N,)
8. `pred_mcap = last_mcap * exp(cumulative_logret)` → rank → compare

In [ ]:
def run_year(year, rank_date, log_returns, mcaps, calendar,
             train_days=TRAIN_DAYS, steps=STEPS,
             max_k=MAX_K, max_var_lags=MAX_VAR_LAGS,
             min_coverage=MIN_COVERAGE, min_obs_mcap=MIN_OBS_MCAP):
    """
    Full PCA + VAR pipeline for one rebalance year.
    Returns a comparison DataFrame (stocks × metrics) or None if skipped.
    """
    # ── 1. Training window ────────────────────────────────────────────────────
    ret_all    = log_returns[log_returns.index <= rank_date]
    ret_train  = ret_all.iloc[-train_days:]

    after = mcaps[mcaps.index > rank_date]
    if len(after) < steps:
        return None   # not enough post-rank data

    # ── 2. Coverage filter ────────────────────────────────────────────────────
    coverage = ret_train.notna().mean()
    tickers  = coverage[coverage >= min_coverage].index
    if len(tickers) < max_k + 5:
        return None   # degenerate
    ret_filt = ret_train[tickers].fillna(0.0)  # T × N_filtered

    # ── 3. Demean ─────────────────────────────────────────────────────────────
    X = ret_filt.values - ret_filt.mean().values  # T × N, zero-mean columns

    # ── 4. Bai-Ng IC1 → K* ───────────────────────────────────────────────────
    K_star = bai_ng_ic1(X, max_k)

    # ── 5. PCA with K* components ─────────────────────────────────────────────
    pca    = PCA(n_components=K_star)
    scores = pca.fit_transform(X)     # T × K*
    loads  = pca.components_          # K* × N
    var_explained = pca.explained_variance_ratio_.sum()

    # ── 6. VAR(p*) forecast ───────────────────────────────────────────────────
    fc_scores, p_star = fit_var_and_forecast(scores, max_var_lags, steps)
    # fc_scores: shape (steps, K*)

    # ── 7. Reconstruct cumulative log-returns ─────────────────────────────────
    # (steps × K*) @ (K* × N) → (steps × N); sum over time → (N,)
    cum_logret = (fc_scores @ loads).sum(axis=0)   # shape (N,) = N filtered tickers
    cum_logret_s = pd.Series(cum_logret, index=tickers)

    # ── 8. Predicted market cap ───────────────────────────────────────────────
    last_mcap  = mcaps[mcaps.index <= rank_date].iloc[-1]
    # For tickers that appear in PCA, use factor-based prediction
    pred_mcap  = last_mcap.reindex(tickers) * np.exp(cum_logret_s)
    pred_s     = pred_mcap.dropna()

    # ── 9. Actual market cap at target date ───────────────────────────────────
    actual_s    = after.iloc[steps - 1].reindex(pred_s.index)
    target_date = after.index[steps - 1]

    cmp = pd.DataFrame({'predicted': pred_s, 'actual': actual_s}).dropna()
    if len(cmp) < 100:
        return None

    cmp['pred_rank']   = cmp['predicted'].rank(ascending=False)
    cmp['actual_rank'] = cmp['actual'].rank(ascending=False)
    cmp['rank_error']  = (cmp['pred_rank'] - cmp['actual_rank']).abs()

    # ── 10. Prior-year snapshot for add/remove analysis ───────────────────────
    if (year - 1) in calendar.index:
        prior_snap       = mcaps[mcaps.index <= calendar.loc[year - 1, 'Rank_Date']].iloc[-1]
        cmp['prev_rank'] = prior_snap.reindex(cmp.index).rank(ascending=False)
    else:
        cmp['prev_rank'] = np.nan

    # Metadata columns
    cmp['year']          = year
    cmp['target_date']   = target_date
    cmp['K_star']        = K_star
    cmp['p_star']        = p_star
    cmp['var_explained'] = var_explained
    cmp['n_train']       = len(ret_train)
    cmp['n_pca_tickers'] = len(tickers)

    return cmp


# ── Russell bands (identical to notebooks 02/03) ──────────────────────────────
RUSSELL_BANDS = {
    'Russell 1000':     (1,    1000),
    'Russell 2000':     (1001, 3000),
    'Russell 3000':     (1,    3000),
    'Russell Microcap': (2001, 4000),
}

def in_band(rank_col, lo, hi):
    return (rank_col >= lo) & (rank_col <= hi)


def assign_banded_membership(mcap_series, prev_rank_series):
    """
    Assign Russell index membership using FTSE Russell cumulative market cap
    percentile banding (Sections 6.10–6.11, Russell US Indexes Methodology).

    Existing members within ±2.5 pct-points of the R1000/R2000 cumulative mcap
    breakpoint stay in their current index. The Microcap boundary (rank 2000)
    uses a tighter ±0.5 pct-point band. New members (prev_rank NaN or outside
    all indexes) are assigned by hard rank cutoffs only. No banding at the
    bottom of the Russell 3000 (rank 3000) or Russell 3000E (rank 4000).
    """
    rk = mcap_series.rank(ascending=False, method='first')
    in_r3e = rk <= 4000

    r3e_sorted = mcap_series[in_r3e].sort_values(ascending=False)
    total_mcap  = r3e_sorted.sum()
    cum_pct     = r3e_sorted.cumsum() / total_mcap * 100   # 0–100 scale

    r3e_ranks = rk[in_r3e]
    idx_1000 = r3e_ranks[r3e_ranks == 1000].index
    idx_2000 = r3e_ranks[r3e_ranks == 2000].index
    bp1000   = float(cum_pct.loc[idx_1000[0]]) if len(idx_1000) else 100.0
    bp2000   = float(cum_pct.loc[idx_2000[0]]) if len(idx_2000) else 100.0

    pr           = prev_rank_series.reindex(mcap_series.index)
    was_r1000    = (pr >= 1)    & (pr <= 1000)
    was_r2000    = (pr >= 1001) & (pr <= 3000)
    was_microcap = (pr >= 2001) & (pr <= 4000)

    pc = cum_pct.reindex(mcap_series.index)

    # Base assignment by hard rank
    mbr = pd.Series(None, index=mcap_series.index, dtype=object)
    mbr[in_r3e & (rk <= 1000)]               = 'Russell 1000'
    mbr[in_r3e & (rk > 1000) & (rk <= 3000)] = 'Russell 2000'
    mbr[in_r3e & (rk > 3000)]                = 'Russell Microcap'

    # R1000/R2000 boundary banding: ±2.5 pct-points around rank-1000 percentile
    in_r1r2_band = in_r3e & pc.notna() & (pc >= bp1000 - 2.5) & (pc <= bp1000 + 2.5)
    mbr[in_r1r2_band & was_r1000] = 'Russell 1000'
    mbr[in_r1r2_band & was_r2000] = 'Russell 2000'

    # Microcap boundary banding: ±0.5 pct-points around rank-2000 percentile
    in_mc_band = in_r3e & pc.notna() & (pc >= bp2000 - 0.5) & (pc <= bp2000 + 0.5) & (rk <= 3000)
    mbr[in_mc_band & was_r2000]    = 'Russell 2000'
    mbr[in_mc_band & was_microcap] = 'Russell Microcap'

    return mbr


---

## 4. Main Loop — All Rebalance Years (2006–2025)

Results are cached per year to `data/processed/pca_var_cache/year_{YEAR}.pkl`. Re-running after interruption loads cached years instantly.

The PCA + VAR fit is relatively fast (seconds per year) because we only fit one VAR on K* ≤ 30 dimensions rather than ~11,000 univariate ARIMA models.

In [ ]:
year_results = {}

for year, row in calendar.iterrows():
    rank_date  = row['Rank_Date']
    cache_file = CACHE_DIR / f'year_{year}.pkl'

    # ── Load from cache ───────────────────────────────────────────────────────
    if cache_file.exists():
        with open(cache_file, 'rb') as f:
            cmp = pickle.load(f)
        year_results[year] = cmp
        rho, _ = spearmanr(cmp['predicted'], cmp['actual'])
        mae    = cmp['rank_error'].mean()
        K_     = cmp['K_star'].iloc[0]
        p_     = cmp['p_star'].iloc[0]
        print(f'[{year}] loaded from cache  K*={K_}  p*={p_}  n={len(cmp):,}  ρ={rho:.3f}  MAE={mae:.0f}')
        continue

    # ── Fit and evaluate ──────────────────────────────────────────────────────
    cmp = run_year(year, rank_date, log_returns, mcaps, calendar)

    if cmp is None:
        print(f'[{year}] skipped (insufficient data)')
        continue

    # ── Cache to disk ─────────────────────────────────────────────────────────
    with open(cache_file, 'wb') as f:
        pickle.dump(cmp, f)

    year_results[year] = cmp

    rho, _ = spearmanr(cmp['predicted'], cmp['actual'])
    mae    = cmp['rank_error'].mean()
    K_     = cmp['K_star'].iloc[0]
    p_     = cmp['p_star'].iloc[0]
    ve_    = cmp['var_explained'].iloc[0]
    print(f'[{year}]  rank_date={rank_date.date()}  target={cmp["target_date"].iloc[0].date()}  '
          f'K*={K_}  p*={p_}  var_expl={ve_:.1%}  n={len(cmp):,}  ρ={rho:.3f}  MAE={mae:.0f}')

print(f'\nCompleted {len(year_results)} / {len(calendar)} years.')

In [ ]:
# ── Apply Russell percentile banding to membership assignments ─────────────
# Uses prior-year rank as proxy for prior membership. Existing members within
# ±2.5 pct-points of the R1000/R2000 cumulative mcap breakpoint stay put.
for year, cmp in year_results.items():
    cmp['pred_membership']   = assign_banded_membership(cmp['predicted'], cmp['prev_rank'])
    cmp['actual_membership'] = assign_banded_membership(cmp['actual'],    cmp['prev_rank'])

print(f'Banding applied to {len(year_results)} years.')


---

## 5. Results

### 5.1 Per-Year Summary Table

Same layout as notebook 02, with two additional columns: **K\*** (Bai-Ng selected factors) and **p\*** (VAR lag order), plus **Var Expl** (fraction of training return variance explained by K* factors).

In [ ]:
rows = []
for year, cmp in year_results.items():
    rho, pval = spearmanr(cmp['predicted'], cmp['actual'])
    rows.append({
        'Year':        year,
        'Rank Date':   calendar.loc[year, 'Rank_Date'].date(),
        'Target Date': cmp['target_date'].iloc[0].date(),
        'Train Days':  cmp['n_train'].iloc[0],
        'N PCA Tkrs':  cmp['n_pca_tickers'].iloc[0],
        'N Stocks':    len(cmp),
        'K*':          cmp['K_star'].iloc[0],
        'p*':          cmp['p_star'].iloc[0],
        'Var Expl':    f"{cmp['var_explained'].iloc[0]:.1%}",
        'Spearman ρ':  round(rho, 4),
        'MAE Rank':    round(cmp['rank_error'].mean(), 1),
        'Median Err':  round(cmp['rank_error'].median(), 1),
    })

summary = pd.DataFrame(rows).set_index('Year')
summary

### 5.2 Spearman ρ and MAE Rank by Year

Same dual bar chart as notebooks 02/03. The orange dashed lines at the notebook 02 AR(10) means are included for reference.

In [ ]:
years = summary.index.tolist()
rhos  = [spearmanr(year_results[y]['predicted'], year_results[y]['actual'])[0] for y in years]
maes  = [year_results[y]['rank_error'].mean() for y in years]

# AR(10) baseline means from notebook 02 (hardcoded for comparison)
AR10_MEAN_RHO = 0.985
AR10_MEAN_MAE = 200.0

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# — Spearman ρ —
ax = axes[0]
ax.bar(years, rhos, color='steelblue', edgecolor='white', zorder=3)
ax.axhline(1,             color='green',  lw=0.8, linestyle='--', label='Perfect (ρ = 1)', zorder=2)
ax.axhline(np.mean(rhos), color='orange', lw=1.2, linestyle=':',
           label=f'PCA+VAR mean ρ = {np.mean(rhos):.3f}', zorder=2)
ax.axhline(AR10_MEAN_RHO, color='gray',   lw=1.0, linestyle='--',
           label=f'AR(10) baseline mean ρ = {AR10_MEAN_RHO:.3f}', zorder=2)
ax.set_ylabel('Spearman ρ', fontsize=11)
ax.set_title('PCA+VAR Market Cap Rank Prediction — Spearman ρ by Year  (60-day horizon)', fontsize=12)
ax.set_ylim(-0.05, 1.10)
ax.legend(fontsize=9)
for x, v in zip(years, rhos):
    ax.text(x, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=7.5)

# — MAE Rank —
ax = axes[1]
ax.bar(years, maes, color='darkorange', edgecolor='white', zorder=3)
ax.axhline(np.mean(maes), color='steelblue', lw=1.2, linestyle=':',
           label=f'PCA+VAR mean MAE = {np.mean(maes):.0f}', zorder=2)
ax.axhline(AR10_MEAN_MAE, color='gray',      lw=1.0, linestyle='--',
           label=f'AR(10) baseline mean MAE = {AR10_MEAN_MAE:.0f}', zorder=2)
ax.set_ylabel('Mean Absolute Rank Error', fontsize=11)
ax.set_xlabel('Rebalance Year', fontsize=11)
ax.set_title('Mean Absolute Rank Error by Year', fontsize=12)
ax.legend(fontsize=9)
for x, v in zip(years, maes):
    ax.text(x, v + 1, f'{v:.0f}', ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.show()

### 5.3 Russell Index Membership Overlap

For each Russell index and each year: of the stocks that *actually* ended up in the index at the 60-day target date, what fraction did our predicted ranks also place there?

In [ ]:
overlap_rows = []
for year, cmp in year_results.items():
    for name, (lo, hi) in RUSSELL_BANDS.items():
        if name == 'Russell 3000':
            pred_in   = set(cmp[cmp['pred_membership'].isin(['Russell 1000', 'Russell 2000'])].index)
            actual_in = set(cmp[cmp['actual_membership'].isin(['Russell 1000', 'Russell 2000'])].index)
        else:
            pred_in   = set(cmp[cmp['pred_membership'] == name].index)
            actual_in = set(cmp[cmp['actual_membership'] == name].index)
        n_actual  = len(actual_in)
        correct   = len(pred_in & actual_in)
        overlap_rows.append({
            'Year':        year,
            'Index':       name,
            'N_Actual':    n_actual,
            'Overlap_Pct': correct / n_actual * 100 if n_actual > 0 else 0,
        })

overlap_df = pd.DataFrame(overlap_rows)

fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharey=True)
for ax, (name, _) in zip(axes.flatten(), RUSSELL_BANDS.items()):
    sub     = overlap_df[overlap_df['Index'] == name]
    mean_ov = sub['Overlap_Pct'].mean()
    ax.bar(sub['Year'], sub['Overlap_Pct'], color='steelblue', edgecolor='white', zorder=3)
    ax.axhline(100,     color='green',  lw=0.8, linestyle='--', label='Perfect (100%)', zorder=2)
    ax.axhline(mean_ov, color='orange', lw=1.2, linestyle=':',  label=f'Mean = {mean_ov:.1f}%', zorder=2)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Year')
    ax.set_ylabel('Membership Overlap (%)')
    ax.set_ylim(0, 112)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=8)

plt.suptitle(
    'Russell Index Membership Overlap: Predicted vs Actual\n'
    '(PCA+VAR, Bai-Ng K*, 60-day horizon)\n'
    '[Membership uses Russell percentile banding rules]',
    fontsize=13
)
plt.tight_layout()
plt.show()


### 5.4 Add/Remove Prediction — Russell 2000

Precision and recall for stocks predicted to enter/exit the Russell 2000, relative to the prior year's rank-date membership as the pre-rebalance baseline.

In [ ]:
# Ground truth: official Russell reconstitution events (not rank-based proxy)
events = pd.read_csv(PROCESSED_DIR / 'russell_events.csv')

rebal_rows = []
for year, cmp in year_results.items():
    if cmp['prev_rank'].isna().all():
        continue

    yr_adds    = set(events[(events['Year'] == year) & (events['Event_Type'] == 'ADD')   ]['Ticker'])
    yr_deletes = set(events[(events['Year'] == year) & (events['Event_Type'] == 'DELETE')]['Ticker'])
    cmp_idx    = set(cmp.index)

    for name, (lo, hi) in RUSSELL_BANDS.items():
        prev_in = in_band(cmp['prev_rank'], lo, hi)

        if name == 'Russell 3000':
            pred_in = cmp['pred_membership'].isin(['Russell 1000', 'Russell 2000'])
        else:
            pred_in = cmp['pred_membership'] == name

        pred_adds_set    = set(cmp[(~prev_in) & pred_in].index)
        pred_removes_set = set(cmp[prev_in & (~pred_in)].index)

        if name == 'Russell 2000':
            # Official Russell events as ground truth — eliminates rank proxy bias
            actual_adds_set    = yr_adds    & cmp_idx
            actual_removes_set = yr_deletes & cmp_idx
        else:
            if name == 'Russell 3000':
                actual_in = cmp['actual_membership'].isin(['Russell 1000', 'Russell 2000'])
            else:
                actual_in = cmp['actual_membership'] == name
            actual_adds_set    = set(cmp[(~prev_in) & actual_in].index)
            actual_removes_set = set(cmp[prev_in & (~actual_in)].index)

        tp_adds    = len(pred_adds_set    & actual_adds_set)
        tp_removes = len(pred_removes_set & actual_removes_set)
        n_aa = len(actual_adds_set);    n_pa = len(pred_adds_set)
        n_ar = len(actual_removes_set); n_pr = len(pred_removes_set)

        rebal_rows.append({
            'Year':             year,
            'Index':            name,
            'Actual Adds':      n_aa,
            'Add_Precision':    tp_adds    / n_pa if n_pa > 0 else np.nan,
            'Add_Recall':       tp_adds    / n_aa if n_aa > 0 else np.nan,
            'Actual Removes':   n_ar,
            'Remove_Precision': tp_removes / n_pr if n_pr > 0 else np.nan,
            'Remove_Recall':    tp_removes / n_ar if n_ar > 0 else np.nan,
        })

rebal_df = pd.DataFrame(rebal_rows)
r2k = rebal_df[rebal_df['Index'] == 'Russell 2000'].set_index('Year')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (metric_p, metric_r, label) in zip(axes, [
    ('Add_Precision',    'Add_Recall',    'Adds'),
    ('Remove_Precision', 'Remove_Recall', 'Removes'),
]):
    w = 0.35
    x = np.arange(len(r2k))
    ax.bar(x - w/2, r2k[metric_p] * 100, w, label='Precision', color='steelblue',  alpha=0.85, zorder=3)
    ax.bar(x + w/2, r2k[metric_r] * 100, w, label='Recall',    color='darkorange', alpha=0.85, zorder=3)
    ax.axhline(r2k[metric_p].mean() * 100, color='steelblue',  lw=1.3, linestyle=':',
               label=f'Avg Precision = {r2k[metric_p].mean():.0%}', zorder=2)
    ax.axhline(r2k[metric_r].mean() * 100, color='darkorange', lw=1.3, linestyle=':',
               label=f'Avg Recall = {r2k[metric_r].mean():.0%}', zorder=2)
    ax.set_xticks(x)
    ax.set_xticklabels(r2k.index, rotation=45)
    ax.set_ylabel('%', fontsize=11)
    ax.set_ylim(0, 112)
    ax.set_title(f'Russell 2000 {label}: Precision & Recall by Year', fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle(
    'Russell 2000 Membership Change Prediction  (PCA+VAR, Bai-Ng K*, 60-day horizon)\n'
    'Ground truth = official Russell ADD/DELETE events (russell_events.csv)',
    fontsize=12
)
plt.tight_layout()
plt.show()

print('Russell 2000 Add/Remove Summary (mean ± std across years):')
print(r2k[['Actual Adds', 'Add_Precision', 'Add_Recall',
           'Actual Removes', 'Remove_Precision', 'Remove_Recall']]
      .agg(['mean', 'std'])
      .applymap(lambda v: f'{v:.1%}' if abs(v) < 10 else f'{v:.0f}'))


### 5.5 Pooled Rank Error Distribution

All stock-years pooled. The CDF and ρ histogram mirror notebooks 02/03 exactly.

In [ ]:
all_errors  = pd.concat([cmp['rank_error'] for cmp in year_results.values()])
all_pred    = pd.concat([cmp['predicted']  for cmp in year_results.values()])
all_actual  = pd.concat([cmp['actual']     for cmp in year_results.values()])

pooled_rho, _ = spearmanr(all_pred, all_actual)
pooled_mae    = all_errors.mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# — CDF —
ax = axes[0]
sorted_err = np.sort(all_errors.values)
cdf = np.arange(1, len(sorted_err) + 1) / len(sorted_err) * 100
ax.plot(sorted_err, cdf, color='steelblue', lw=2)
for thresh, ypos in [(50, 12), (100, 20), (250, 30), (500, 40)]:
    pct = (all_errors <= thresh).mean() * 100
    ax.axvline(thresh, color='gray', linestyle='--', lw=0.8)
    ax.text(thresh + 10, ypos, f'{pct:.0f}% ≤ {thresh}', fontsize=8.5, color='dimgray')
ax.set_xlabel('Absolute Rank Error', fontsize=11)
ax.set_ylabel('Cumulative % of Stock-Years', fontsize=11)
ax.set_title(
    f'CDF of Rank Errors — All Years Pooled\n'
    f'n = {len(all_errors):,} stock-years  |  MAE = {pooled_mae:.0f}  |  Spearman ρ = {pooled_rho:.4f}',
    fontsize=11
)
ax.set_ylim(0, 100)

# — Per-year ρ histogram —
ax = axes[1]
rho_vals = [spearmanr(cmp['predicted'], cmp['actual'])[0] for cmp in year_results.values()]
ax.hist(rho_vals, bins=10, color='steelblue', edgecolor='white', zorder=3)
ax.axvline(np.mean(rho_vals), color='darkorange', lw=1.5, linestyle='--',
           label=f'Mean ρ = {np.mean(rho_vals):.3f}')
ax.set_xlabel('Spearman ρ', fontsize=11)
ax.set_ylabel('Number of Years', fontsize=11)
ax.set_title('Distribution of Per-Year Spearman ρ', fontsize=11)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Pooled Spearman ρ  : {pooled_rho:.4f}')
print(f'Pooled MAE rank    : {pooled_mae:.1f}')
print(f'Total stock-years  : {len(all_errors):,}')

---

## 6. PCA / VAR Diagnostics

### 6.1 Selected K* and p* by Year

How does the complexity of the factor structure shift across market regimes? More volatile years may warrant more factors (higher K*) if the idiosyncratic structure changes, or fewer if a dominant market shock concentrates variance in PC1.

In [ ]:
meta = pd.DataFrame({
    'Year':        list(year_results.keys()),
    'K_star':      [cmp['K_star'].iloc[0]        for cmp in year_results.values()],
    'p_star':      [cmp['p_star'].iloc[0]         for cmp in year_results.values()],
    'var_explained': [cmp['var_explained'].iloc[0] for cmp in year_results.values()],
    'n_pca_tickers': [cmp['n_pca_tickers'].iloc[0] for cmp in year_results.values()],
}).set_index('Year')

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

# K*
ax = axes[0]
ax.bar(meta.index, meta['K_star'], color='steelblue', edgecolor='white', zorder=3)
ax.axhline(meta['K_star'].mean(), color='orange', lw=1.2, linestyle=':',
           label=f'Mean K* = {meta["K_star"].mean():.1f}')
ax.set_ylabel('K* (Bai-Ng selected factors)', fontsize=10)
ax.set_title('Bai-Ng IC1 Selected Number of Factors K* by Year', fontsize=11)
ax.legend(fontsize=9)
for x, v in zip(meta.index, meta['K_star']):
    ax.text(x, v + 0.1, str(v), ha='center', va='bottom', fontsize=8)

# VAR lag p*
ax = axes[1]
ax.bar(meta.index, meta['p_star'], color='darkorange', edgecolor='white', zorder=3)
ax.axhline(meta['p_star'].mean(), color='steelblue', lw=1.2, linestyle=':',
           label=f'Mean p* = {meta["p_star"].mean():.1f}')
ax.set_ylabel('p* (VAR lag order)', fontsize=10)
ax.set_title('VAR BIC Selected Lag Order p* by Year', fontsize=11)
ax.legend(fontsize=9)
for x, v in zip(meta.index, meta['p_star']):
    ax.text(x, v + 0.05, str(v), ha='center', va='bottom', fontsize=8)

# Variance explained
ax = axes[2]
ax.bar(meta.index, meta['var_explained'] * 100, color='mediumseagreen', edgecolor='white', zorder=3)
ax.axhline(meta['var_explained'].mean() * 100, color='orange', lw=1.2, linestyle=':',
           label=f'Mean = {meta["var_explained"].mean():.1%}')
ax.set_ylabel('Variance Explained by K* Factors (%)', fontsize=10)
ax.set_xlabel('Rebalance Year', fontsize=11)
ax.set_title('Fraction of Training Return Variance Explained by K* Factors', fontsize=11)
ax.legend(fontsize=9)
for x, v in zip(meta.index, meta['var_explained'] * 100):
    ax.text(x, v + 0.3, f'{v:.0f}%', ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.show()

print(meta.to_string())

### 6.2 Scree Plot — Representative Year (2019)

Full scree for the 2019 training window: eigenvalue decay and cumulative variance. The vertical dashed line marks K* selected by Bai-Ng IC1. This illustrates how the IC trades off fit improvement against complexity.

In [ ]:
SCREE_YEAR = 2019
rank_date_scree = calendar.loc[SCREE_YEAR, 'Rank_Date']

# Rebuild the training matrix for the scree year
ret_sc   = log_returns[log_returns.index <= rank_date_scree].iloc[-TRAIN_DAYS:]
cov_sc   = ret_sc.notna().mean()
tkr_sc   = cov_sc[cov_sc >= MIN_COVERAGE].index
X_sc     = ret_sc[tkr_sc].fillna(0.0).values
X_sc     = X_sc - X_sc.mean(axis=0)

n_comp_scree = min(50, X_sc.shape[0] - 1, X_sc.shape[1] - 1)
pca_scree    = PCA(n_components=n_comp_scree)
pca_scree.fit(X_sc)

evr      = pca_scree.explained_variance_ratio_
cum_evr  = np.cumsum(evr)
K_scree  = year_results[SCREE_YEAR]['K_star'].iloc[0] if SCREE_YEAR in year_results else bai_ng_ic1(X_sc, MAX_K)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree (individual variance)
ax = axes[0]
ax.bar(range(1, n_comp_scree + 1), evr * 100, color='steelblue', edgecolor='white')
ax.axvline(K_scree, color='red', lw=1.5, linestyle='--', label=f'K* = {K_scree} (Bai-Ng IC1)')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title(f'Scree Plot — {SCREE_YEAR} Training Window\n({len(tkr_sc):,} tickers, {TRAIN_DAYS} days)')
ax.legend(fontsize=9)

# Cumulative variance
ax = axes[1]
ax.plot(range(1, n_comp_scree + 1), cum_evr * 100, marker='o', markersize=4,
        color='steelblue', lw=2)
ax.axvline(K_scree, color='red', lw=1.5, linestyle='--', label=f'K* = {K_scree}')
ax.axhline(cum_evr[K_scree - 1] * 100, color='red', lw=0.8, linestyle=':')
ax.text(K_scree + 0.5, cum_evr[K_scree - 1] * 100 - 3,
        f'{cum_evr[K_scree - 1]:.1%}', color='red', fontsize=9)
for pct in [50, 70, 90]:
    ax.axhline(pct, color='gray', lw=0.6, linestyle=':')
    ax.text(n_comp_scree - 1, pct + 0.5, f'{pct}%', color='gray', fontsize=8)
ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Variance Explained (%)')
ax.set_title(f'Cumulative Variance — {SCREE_YEAR}')
ax.set_ylim(0, 105)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'{SCREE_YEAR}: K* = {K_scree}, cumulative variance explained = {cum_evr[K_scree - 1]:.1%}')

### 6.3 PC1 Score vs Market Return

A sanity check: in a well-specified factor model, PC1 should be almost perfectly correlated with the equal-weighted daily market return. Deviations indicate the training cross-section is dominated by some other source of co-movement (e.g. in crisis years, credit/liquidity risk may matter more than size).

In [ ]:
# Rebuild scores for the scree year to show PC1 vs equal-weighted return
pca_check    = PCA(n_components=1)
scores_check = pca_check.fit_transform(X_sc)[:, 0]

ew_return = ret_sc[tkr_sc].fillna(0.0).mean(axis=1).values

# Align sign: PC1 may be negatively correlated with market return
if np.corrcoef(scores_check, ew_return)[0, 1] < 0:
    scores_check = -scores_check

rho_check, _ = spearmanr(scores_check, ew_return)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(ew_return, scores_check, alpha=0.25, s=8, color='steelblue')
ax.set_xlabel('Equal-Weighted Daily Market Return')
ax.set_ylabel('PC1 Score')
ax.set_title(
    f'PC1 Score vs Equal-Weighted Market Return — {SCREE_YEAR} Training Window\n'
    f'Spearman ρ = {rho_check:.4f}  (perfect factor = 1.0)'
)
plt.tight_layout()
plt.show()